# Lab 2.3 - Prompting Techniques and Generative Tasks (Solution)
**Day 2 - Session 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/lab2_3_prompting/solution/lab2_3_prompting.ipynb)

**Goal:** Apply prompting techniques to three generative tasks: summarization,
code generation, and text labeling. All LLM calls go through the shared client
(`get_completion`) using the cheap `mini` tier.

This solution has the full working code. It needs an LLM API key (set in
Colab or your local `.env`). Runs on CPU; no GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q litellm python-dotenv

## Setup: make the shared client available
This finds the `shared/` package locally, or clones the companion repo when
running on Colab. It also makes sure an API key is set.

In [ ]:
import os
import subprocess
import sys


def _find_repo_root():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "shared")):
            return d
        d = os.path.dirname(d)
    return None


root = _find_repo_root()
if root is None:  # On Colab: clone the companion repo for the shared client.
    if not os.path.isdir("gen-ai-companion-intuite"):
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/kpassoubady/gen-ai-companion-intuite.git"],
            check=True,
        )
    root = "gen-ai-companion-intuite"
sys.path.insert(0, root)

from shared import get_completion, show_config

# Ensure an API key is present. Locally it loads from .env on import above.
# On Colab, use a stored secret or type it once.
if not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "GEMINI_API_KEY",
                                       "ANTHROPIC_API_KEY", "AZURE_API_KEY")):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
    os.environ.setdefault("LLM_PROVIDER", "openai")

show_config()

## Part 1. Summarization
Start zero-shot, then refine with a role and an output format. Watch how each
change tightens the result.

In [ ]:
ARTICLE = """
Retrieval-augmented generation (RAG) combines a language model with a search
step. Before answering, the system retrieves relevant documents from a knowledge
base and adds them to the prompt. This grounds the answer in real data, reduces
hallucination, and lets the model use private or up-to-date information it never
saw during training. A typical pipeline chunks documents, embeds them, stores
the vectors, retrieves the closest chunks for a query, and passes them to the
model as context.
"""

# Zero-shot summarization
zero_shot = get_completion(
    messages=[{"role": "user", "content": f"Summarize this text:\n{ARTICLE}"}],
    tier="mini",
    temperature=0.0,
)
print("Zero-shot summary:\n", zero_shot)

In [ ]:
# Refined: add a role and a strict output format
refined = get_completion(
    messages=[
        {"role": "system", "content":
            "You are a technical editor. Summarize for a busy engineer."},
        {"role": "user", "content":
            f"Summarize the text in exactly 3 bullet points.\n{ARTICLE}"},
    ],
    tier="mini",
    temperature=0.0,
)
print("Refined summary:\n", refined)

## Part 2. Code generation
Give a clear spec and ask for constraints (type hints, a docstring). Specific
instructions produce more usable code.

In [ ]:
code = get_completion(
    messages=[
        {"role": "system", "content":
            "You are a senior Python developer. Return only code, no prose."},
        {"role": "user", "content":
            "Write a function `is_palindrome(text: str) -> bool` that ignores "
            "case and non-alphanumeric characters. Include type hints and a "
            "docstring."},
    ],
    tier="mini",
    temperature=0.0,
)
print(code)

## Part 3. Text labeling
Classify support tickets. Compare zero-shot with few-shot, where examples pin
down the exact label set and edge cases.

In [ ]:
TICKETS = [
    "I was charged twice for my subscription this month.",
    "The mobile app crashes every time I open the reports tab.",
    "How do I add a new user to my team account?",
]

# Zero-shot labeling
for ticket in TICKETS:
    label = get_completion(
        messages=[
            {"role": "system", "content":
                "Label the ticket as BILLING, BUG, or HOWTO. Reply with one word."},
            {"role": "user", "content": ticket},
        ],
        tier="mini",
        temperature=0.0,
    )
    print(f"{label.strip():>8}  <-  {ticket}")

In [ ]:
# Few-shot labeling: examples make the label set and format explicit
FEWSHOT = [
    {"role": "system", "content":
        "Label each ticket as BILLING, BUG, or HOWTO. Reply with one word."},
    {"role": "user", "content": "My card was declined at checkout."},
    {"role": "assistant", "content": "BILLING"},
    {"role": "user", "content": "The export button does nothing when clicked."},
    {"role": "assistant", "content": "BUG"},
    {"role": "user", "content": "Where do I change my password?"},
    {"role": "assistant", "content": "HOWTO"},
]

for ticket in TICKETS:
    label = get_completion(
        messages=FEWSHOT + [{"role": "user", "content": ticket}],
        tier="mini",
        temperature=0.0,
    )
    print(f"{label.strip():>8}  <-  {ticket}")

## Stretch goals
1. Add chain-of-thought to the code task: ask the model to plan before coding.
2. Ask for JSON labels (`{"category": ..., "confidence": ...}`) and parse them.

### Stretch 2: structured JSON output

In [ ]:
import json

raw = get_completion(
    messages=[
        {"role": "system", "content":
            'Label the ticket. Reply with JSON only: '
            '{"category": "BILLING|BUG|HOWTO", "confidence": 0-1}'},
        {"role": "user", "content": TICKETS[0]},
    ],
    tier="mini",
    temperature=0.0,
)
print("Raw:", raw)
parsed = json.loads(raw)
print("Parsed category:", parsed["category"])

## Expected output
- The refined summary is tighter than the zero-shot one and uses 3 bullets.
- The generated `is_palindrome` function has type hints and a docstring.
- Zero-shot and few-shot both label the three tickets as BILLING, BUG, HOWTO.
- The stretch cell returns valid JSON that parses into a category.